In [1]:
##### naprawa + deduplikacja

import re
import pandas as pd

input_file  = 'hathi_articles.csv'
output_file = 'hathi_articles_fixed.csv'

## wyciąganie wszystkich poprawnych rekordów

good_lines = []
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        ### poprawny rekord: zaczyna się od słowa (keyword) + przecinek + cudzysłów
        if re.match(r'^[a-zA-Z][^,]*?,"', line):
            ### naprawia tylko cudzysłowy (jeśli były podmienione)
            line = line.replace("'", '"')
            good_lines.append(line)

print(f"Znaleziono {len(good_lines):,} poprawnych rekordów")

## zapis jako czysty CSV – używa engine='python' + quoting żeby nie wywalało się na przecinkach
import csv

with open(output_file, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['keyword','title','authors','publisher','published_date','description','hathi_id'])
    for line in good_lines:
        original_line = line
        ### usuwa wszystkie tabulatory i nadmiarowe cudzysłowy
        line = line.replace('\t', ' ').replace('"""', '"').replace('""', '"')

        #### wycina tylko rekordy gdzie śmieci (loc.ark, mdp., deny ic itd.) są przed ostatnim przecinkiem
        #### czyli w tytule/autorze/imprint – a nie w htid (ostatnie pole)
        if ',' in line:
            before_last_comma = line.rsplit(',', 6)[0]   #### wszystko przed htid
            if any(bad in before_last_comma for bad in ['loc.ark:/', 'deny ic', 'allow pd', 'uva.', 'mdp.', 'hvd.', 'uc1.', 'nyp.']):
                continue

        ### rozbicie tylko po pierwszych 6 przecinkach (bo title może mieć przecinki)
        parts = line.split(',', 6)
        if len(parts) == 7:
            ### dodatkowa ochrona: jeśli title zaczyna się od " i kończy na " - obetnij
            if parts[1].startswith('"') and parts[1].endswith('"'):
                parts[1] = parts[1][1:-1]
            writer.writerow(parts)

print("Zapisano tymczasowy czysty plik")

## deduplikacja
df = pd.read_csv(output_file, dtype=str)
before = len(df)
df.drop_duplicates(subset=['title', 'authors'], keep='first', inplace=True)
df.to_csv(output_file, index=False)

print(f"\nWykonane.")
print(f"   Przed deduplikacją: {before:,}")
print(f"   Po deduplikacji:     {len(df):,}")
print(f"   Czysty plik zapisany jako {output_file}")

Znaleziono 50,906 poprawnych rekordów
Zapisano tymczasowy czysty plik

Wykonane.
   Przed deduplikacją: 50,882
   Po deduplikacji:     26,059
   Czysty plik zapisany jako hathi_articles_fixed.csv
